In [40]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np 
import pandas as pd 
import xgboost as xgb
from xgboost.sklearn import XGBClassifier
from sklearn import metrics   #Additional scklearn functions
from sklearn.model_selection import GridSearchCV
import matplotlib.pylab as plt
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 12, 4

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [41]:
education_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_Education_train_set.csv')
education_test = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_Education_test_set.csv')
household_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_HouseholdInfo_train_set.csv')
household_test = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_HouseholdInfo_test_set.csv')
subjective_poverty_train = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/module_SubjectivePoverty_train_set.csv')
sample_submission = pd.read_csv('/Users/callum/University/School Work/3B/STAT 441/Kaggle_vscode/f-2024-kaggle-contest-for-classification/sample_submission.csv')

subjective_poverty_train.head(5)


,psu_hh_idcode,subjective_poverty_1,subjective_poverty_2,subjective_poverty_3,subjective_poverty_4,subjective_poverty_5,subjective_poverty_6,subjective_poverty_7,subjective_poverty_8,subjective_poverty_9,subjective_poverty_10
0,30_8_1,0,0,0,1,0,0,0,0,0,0
1,194_1_2,1,0,0,0,0,0,0,0,0,0
2,224_6_1,0,0,1,0,0,0,0,0,0,0
3,323_10_1,0,0,0,0,1,0,0,0,0,0
4,428_10_1,0,0,0,1,0,0,0,0,0,0


In [42]:
subjective_poverty_train[['psu', 'hh', 'idcode']] = subjective_poverty_train['psu_hh_idcode'].str.split('_', expand=True).astype(int)

train_data = pd.merge(education_train, household_train, on=['psu', 'hh', 'idcode'], how='inner')
train_data = pd.merge(train_data, subjective_poverty_train, on=['psu', 'hh', 'idcode'], how='inner')

print(len(train_data))


# Creating a new column 'Genre' based on a condition
train_data['Sub_Pov_Ranking'] = 0  # Initialize the 'Genre' column with 'Other' as the default value

# Using DataFrame.loc[] to set values based on the condition
train_data.loc[train_data['subjective_poverty_1'] == 1, 'Sub_Pov_Ranking'] = 0
train_data.loc[train_data['subjective_poverty_2'] == 1, 'Sub_Pov_Ranking'] = 1
train_data.loc[train_data['subjective_poverty_3'] == 1, 'Sub_Pov_Ranking'] = 2
train_data.loc[train_data['subjective_poverty_4'] == 1, 'Sub_Pov_Ranking'] = 3
train_data.loc[train_data['subjective_poverty_5'] == 1, 'Sub_Pov_Ranking'] = 4
train_data.loc[train_data['subjective_poverty_6'] == 1, 'Sub_Pov_Ranking'] = 5
train_data.loc[train_data['subjective_poverty_7'] == 1, 'Sub_Pov_Ranking'] = 6
train_data.loc[train_data['subjective_poverty_8'] == 1, 'Sub_Pov_Ranking'] = 7
train_data.loc[train_data['subjective_poverty_9'] == 1, 'Sub_Pov_Ranking'] = 8
train_data.loc[train_data['subjective_poverty_10'] == 1, 'Sub_Pov_Ranking'] = 9

# i=1
# while i <= 10:
#     train_data.drop(['subjective_poverty_' + str(i)], axis=1)
#     i = i + 1

# i = 6
# train_data.drop(['subjective_poverty_' + str(i)], axis=1)


5334


In [43]:
def modelfit(alg, dtrain, predictors,useTrainCV=True, cv_folds=5, early_stopping_rounds=50):
    
    if useTrainCV:
        xgb_param = alg.get_xgb_params()
        xgtrain = xgb.DMatrix(dtrain[predictors].values, label=dtrain[target].values)
        cvresult = xgb.cv(xgb_param, xgtrain, num_boost_round=alg.get_params()['n_estimators'], nfold=cv_folds,
            metrics='auc', early_stopping_rounds=early_stopping_rounds)
        alg.set_params(n_estimators=cvresult.shape[0])
    
    #Fit the algorithm on the data
    alg.fit(dtrain[predictors], dtrain['Sub_Pov_Ranking'],eval_metric='auc')
        
    #Predict training set:
    dtrain_predictions = alg.predict(dtrain[predictors])
    dtrain_predprob = alg.predict_proba(dtrain[predictors])[:,1]
        
    #Print model report:
    print("\nModel Report")
    print("Accuracy : %.4g" % metrics.accuracy_score(dtrain['Sub_Pov_Ranking'].values, dtrain_predictions))
    print("AUC Score (Train): %f" % metrics.roc_auc_score(dtrain['Sub_Pov_Ranking'], dtrain_predprob))
                    
    feat_imp = pd.Series(alg.booster().get_fscore()).sort_values(ascending=False)
    feat_imp.plot(kind='bar', title='Feature Importances')
    plt.ylabel('Feature Importance Score')

In [44]:
target = 'Sub_Pov_Ranking'

train_data['id'] = np.arange(1, len(train_data)+1)


from sklearn.model_selection import train_test_split

train, test = train_test_split(train_data, test_size=0.3)

print(len(train.index))

print(len(test.index))

IDcol = 'id'

non_predictors = ["psu", "hh", 'idcode', "psu_hh_idcode"]


3733
1601


In [45]:
#Choose all predictors except target & IDcols
# predictors = [x for x in train_data.columns if x not in [non_predictors, target, IDcol, x.startswith('subjective_poverty_10')]]
# xgb1 = XGBClassifier(
#  learning_rate =0.1,
#  n_estimators=1000,
#  max_depth=5,
#  min_child_weight=1,
#  gamma=0,
#  subsample=0.8,
#  colsample_bytree=0.8,
#  objective= 'multi:softprob',
#  nthread=4,
#  scale_pos_weight=1,
#  seed=27)
# modelfit(xgb1, train_data, predictors)

selected_columns = [column for column in train.columns if column.startswith('subjective_poverty')]

predictors = [x for x in train.columns if x not in ["psu", "hh", 'idcode', "psu_hh_idcode", IDcol, target] and x not in selected_columns]

X_train = train[predictors]

y_train = train['Sub_Pov_Ranking']

# Create an instance of the XGBClassifier
model = XGBClassifier(objective='multi:softprob', enable_categorical =True)

# Fit the model to the training data
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [83]:
predictors = [x for x in test.columns if x not in ["psu", "hh", 'idcode', "psu_hh_idcode", target, IDcol] and x not in selected_columns]

X_test = test[predictors]

y_test = test[selected_columns]


X_predict = model.predict_proba(X_test)


print(len(X_predict))

print(y_test.iloc[1][1])



# # make predictions for test data
# y_pred = model.predict(X_test)
# predictions = [round(value) for value in y_pred]


# # evaluate predictions
# accuracy = accuracy_score(y_test, predictions)
# print("Accuracy: %.2f%%" % (accuracy * 100.0))

1601
0


/var/folders/g8/vy9w_fxd6r39lbfd4qykf2j80000gn/T/ipykernel_9894/3854082828.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(y_test.iloc[1][1])


In [109]:
import math

def multiclass_log_loss(actual,predicted):
    loss = 0 
    i = 1
    while i < len(actual):
        q = 0
        while q < len(actual.iloc[i]):
            y_ij = actual.iloc[i][q]
            log_p_ij = math.log(predicted[i][q])
            loss += y_ij*log_p_ij
            q = q + 1
        i = i + 1
    return(loss/ -len(actual))

In [110]:
print(multiclass_log_loss(y_test, X_predict))

/var/folders/g8/vy9w_fxd6r39lbfd4qykf2j80000gn/T/ipykernel_9894/1365599809.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  y_ij = actual.iloc[i][q]


2.1228749994058647
